In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("esrakavalci/sust-ddd")

print("Path to dataset files:", path)

100%|██████████| 4.44G/4.44G [00:46<00:00, 101MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/esrakavalci/sust-ddd/versions/2


In [ ]:
import os

# List the contents of the downloaded dataset path
dataset_contents = os.listdir(path)

print(f"Contents of the dataset directory '{path}':")
for item in dataset_contents:
    print(f"- {item}")

Contents of the dataset directory '/root/.cache/kagglehub/datasets/esrakavalci/sust-ddd/versions/2':
- SUST Driver Drowsiness Dataset
- README.md


In [ ]:
import os

dataset_main_dir = os.path.join(path, "SUST Driver Drowsiness Dataset")

if os.path.isdir(dataset_main_dir):
    print(f"\nContents of the main dataset directory '{dataset_main_dir}':")
    main_dir_contents = os.listdir(dataset_main_dir)
    for item in main_dir_contents:
        print(f"- {item}")


Contents of the main dataset directory '/root/.cache/kagglehub/datasets/esrakavalci/sust-ddd/versions/2/SUST Driver Drowsiness Dataset':
- drowsiness
- not drowsiness


In [ ]:
drowsiness_dir = os.path.join(dataset_main_dir, "drowsiness")
if os.path.isdir(drowsiness_dir):
    print(f"\nContents of the 'drowsiness' directory '{drowsiness_dir}':")
    drowsiness_contents = os.listdir(drowsiness_dir)
    # Print only a few items if the directory is very large
    for item in drowsiness_contents[:5]: # Display first 5 items to get an idea
        print(f"- {item}")
    if len(drowsiness_contents) > 5:
        print(f"- ... ({len(drowsiness_contents) - 5} more items)")
    elif len(drowsiness_contents) == 0:
        print("- (empty directory)")

non_drowsiness_dir = os.path.join(dataset_main_dir, "not drowsiness")

if os.path.isdir(non_drowsiness_dir):
    print(f"\nContents of the 'not drowsiness' directory '{non_drowsiness_dir}':")
    non_drowsiness_contents = os.listdir(non_drowsiness_dir)
    # Print only a few items if the directory is very large
    for item in non_drowsiness_contents[:5]: # Display first 5 items to get an idea
        print(f"- {item}")
    if len(non_drowsiness_contents) > 5:
        print(f"- ... ({len(non_drowsiness_contents) - 5} more items)")
    elif len(non_drowsiness_contents) == 0:
        print("- (empty directory)")


Contents of the 'drowsiness' directory '/root/.cache/kagglehub/datasets/esrakavalci/sust-ddd/versions/2/SUST Driver Drowsiness Dataset/drowsiness':
- d_935.mp4
- d_227.mp4
- d_819.mp4
- d_888.mp4
- d_300.mp4
- ... (970 more items)

Contents of the 'not drowsiness' directory '/root/.cache/kagglehub/datasets/esrakavalci/sust-ddd/versions/2/SUST Driver Drowsiness Dataset/not drowsiness':
- n_787.mp4
- n_208.mp4
- n_126.mp4
- n_1030.mp4
- n_13.mp4
- ... (1094 more items)


In [ ]:
import cv2
import numpy as np

IMG_SIZE = (64, 64)
SEQUENCE_LENGTH = 30

def extract_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, IMG_SIZE)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

        # Stop if too many frames
        if len(frames) >= SEQUENCE_LENGTH:
            break

    cap.release()

    # Pad if less than SEQUENCE_LENGTH
    if len(frames) < SEQUENCE_LENGTH:
        missing = SEQUENCE_LENGTH - len(frames)
        frames.extend([np.zeros((64,64,3), dtype=np.uint8)] * missing)

    return np.array(frames)


In [ ]:
import os

drowsy_path = os.path.join(dataset_main_dir, "drowsiness")
not_drowsy_path = os.path.join(dataset_main_dir, "not drowsiness")

X = []
y = []

# Drowsiness = 1
for video in os.listdir(drowsy_path):
    video_file = os.path.join(drowsy_path, video)
    frames = extract_frames(video_file)
    X.append(frames)
    y.append(1)

# Not drowsy = 0
for video in os.listdir(not_drowsy_path):
    video_file = os.path.join(not_drowsy_path, video)
    frames = extract_frames(video_file)
    X.append(frames)
    y.append(0)

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (2074, 30, 64, 64, 3)
y shape: (2074,)


In [ ]:
from sklearn.model_selection import train_test_split

# Split data into training and temporary (validation + test) sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=12)

# Split temporary set into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.25, random_state=35)

# Print the shapes of the resulting datasets
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

X_train shape: (1451, 30, 64, 64, 3), y_train shape: (1451,)
X_val shape: (467, 30, 64, 64, 3), y_val shape: (467,)
X_test shape: (156, 30, 64, 64, 3), y_test shape: (156,)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, TimeDistributed, Conv2D, MaxPooling2D, BatchNormalization
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.optimizers import AdamW

SEQUENCE_LENGTH = 16
IMG_SIZE = (64, 64, 3)
model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, *IMG_SIZE)),
    TimeDistributed(Conv2D(32, (3,3), activation='relu')),
    TimeDistributed(BatchNormalization()),
    TimeDistributed(MaxPooling2D((2,2))),

    TimeDistributed(Conv2D(64, (3,3), activation='relu')),
    TimeDistributed(BatchNormalization()),
    TimeDistributed(MaxPooling2D((2,2))),

    TimeDistributed(Conv2D(128, (3,3), activation='relu', padding='same')),
    TimeDistributed(BatchNormalization()),
    TimeDistributed(MaxPooling2D((2,2))),

    TimeDistributed(GlobalAveragePooling2D()),

    LSTM(128,return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),

    Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
    Dropout(0.4),

    Dense(1, activation='sigmoid')
])

optimizer = AdamW(learning_rate=1e-4, weight_decay=1e-5)

model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 16, 62, 62, 32) │           896 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 16, 62, 62, 32) │           128 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 16, 31, 31, 32) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 16, 29, 29, 64) │        18,496 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, 16, 29, 29, 64) │           256 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_5              │ (None, 16, 14, 14, 64) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_6              │ (None, 16, 14, 14,     │        73,856 │
│ (TimeDistributed)               │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_7              │ (None, 16, 14, 14,     │           512 │
│ (TimeDistributed)               │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_8              │ (None, 16, 7, 7, 128)  │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_9              │ (None, 16, 128)        │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 16, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 279,361 (1.07 MB)

 Trainable params: 278,913 (1.06 MB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
import tensorflow as tf

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_loss",
        save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.4,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=8,   # safe for Colab RAM/GPU
    callbacks=callbacks,
    verbose=1
)


Epoch 1/50
182/182 ━━━━━━━━━━━━━━━━━━━━ 84s 271ms/step - accuracy: 0.5569 - loss: 0.6878 - val_accuracy: 0.6552 - val_loss: 0.6425 - learning_rate: 1.0000e-04
Epoch 2/50
182/182 ━━━━━━━━━━━━━━━━━━━━ 45s 247ms/step - accuracy: 0.6733 - loss: 0.6209 - val_accuracy: 0.6510 - val_loss: 0.6399 - learning_rate: 1.0000e-04
Epoch 3/50
182/182 ━━━━━━━━━━━━━━━━━━━━ 44s 243ms/step - accuracy: 0.7285 - loss: 0.5712 - val_accuracy: 0.7238 - val_loss: 0.5995 - learning_rate: 1.0000e-04
Epoch 4/50
182/182 ━━━━━━━━━━━━━━━━━━━━ 44s 244ms/step - accuracy: 0.7595 - loss: 0.5403 - val_accuracy: 0.7345 - val_loss: 0.5737 - learning_rate: 1.0000e-04
Epoch 5/50
182/182 ━━━━━━━━━━━━━━━━━━━━ 44s 243ms/step - accuracy: 0.7726 - loss: 0.5105 - val_accuracy: 0.7537 - val_loss: 0.5496 - learning_rate: 1.0000e-04
Epoch 6/50
182/182 ━━━━━━━━━━━━━━━━━━━━ 44s 244ms/step - accuracy: 0.7753 - loss: 0.4948 - val_accuracy: 0.7645 - val_loss: 0.5269 - learning_rate: 1.0000e-04
Epoch 7/50
182/182 ━━━━━━━━━━━━━━━━━━━━ 44s 24

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label="Train Acc")
plt.plot(history.history['val_accuracy'], label="Val Acc")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label="Train Loss")
plt.plot(history.history['val_loss'], label="Val Loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Make predictions on the test set
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int) # Convert probabilities to binary predictions

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Compute the confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(4,3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Drowsy (0)', 'Drowsy (1)'],
            yticklabels=['Not Drowsy (0)', 'Drowsy (1)'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
import os
import tensorflow as tf

if os.path.exists("best_model.keras"):
    model = tf.keras.models.load_model("best_model.keras")
    print("✅ Loaded best_model.keras")
else:
    print("✅ Using existing `model` already in notebook")

print("Model input shape:", model.input_shape)  # (None, 16, 64, 64, 3)

In [ ]:
import numpy as np
import cv2, time, base64, io
from PIL import Image
from collections import deque
from google.colab.output import eval_js
from IPython.display import HTML, display
from scipy.io.wavfile import write

# ----------------- Click to enable audio -----------------
display(HTML("""
<button style="padding:10px 14px; font-size:14px; border-radius:10px; cursor:pointer;"
        onclick="this.innerText='Sound enabled ✅';">
  Click once to enable alarm sound 🔊
</button>
"""))

# ----------------- Beep -----------------
def make_beep():
    t = np.linspace(0, 0.6, int(22050 * 0.6), False)
    tone = 0.6 * np.sin(2 * np.pi * 900 * t)
    audio = np.int16(tone * 32767)
    buf = io.BytesIO()
    write(buf, 22050, audio)
    return base64.b64encode(buf.getvalue()).decode()

BEEP = make_beep()

def alarm():
    display(HTML(f"""
    <audio autoplay>
      <source src="data:audio/wav;base64,{BEEP}" type="audio/wav">
    </audio>
    """))

# ----------------- Webcam -----------------
def start_webcam():
    display(HTML("""
    <video id="video" width="420" height="315" autoplay playsinline
           style="border:2px solid #aaa; border-radius:12px;"></video>
    <canvas id="canvas" width="420" height="315" style="display:none;"></canvas>

    <script>
      const video = document.getElementById('video');
      async function startCamera(){
        video.srcObject = await navigator.mediaDevices.getUserMedia({video:true});
        return true;
      }
      async function captureFrame(){
        const c = document.getElementById('canvas');
        const x = c.getContext('2d');
        x.drawImage(video,0,0,c.width,c.height);
        return c.toDataURL('image/jpeg',0.85);
      }
    </script>
    """))
    return eval_js("startCamera()")

def get_frame_rgb(data_url):
    _, b64 = data_url.split(",", 1)
    img = Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB")
    return np.array(img)

# ----------------- Model -----------------
_, SEQ_LEN, H, W, C = model.input_shape
print(f"Model expects: SEQ_LEN={SEQ_LEN}, H={H}, W={W}, C={C}")

buffer = deque(maxlen=SEQ_LEN)

def preprocess(frame_rgb):
    fr = cv2.resize(frame_rgb, (W, H))
    return fr.astype(np.uint8)

# ----------------- Eye detection -----------------
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
eye_cascade  = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")

def eyes_closed(frame_rgb):
    gray = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.2, 5)
    if len(faces) == 0:
        return False
    x,y,wf,hf = faces[0]
    roi = gray[y:y+hf, x:x+wf]
    eyes = eye_cascade.detectMultiScale(roi, 1.2, 6)
    return len(eyes) == 0

# ----------------- Tuning -----------------
CNN_THRESHOLD = 0.5
CNN_STREAK_N  = 8
EYE_STREAK_N  = 8
ALARM_GAP     = 2.0

cnn_streak = 0
eye_streak = 0
last_alarm = 0

start_webcam()

print("\nLIVE DETECTION RUNNING\n")

try:
    while True:

        rgb = get_frame_rgb(eval_js("captureFrame()"))

        # ---- CNN path ----
        fr = preprocess(rgb)
        buffer.append(fr)

        prob = None
        if len(buffer) == SEQ_LEN:
            x = np.expand_dims(np.array(buffer), axis=0)
            prob = float(model.predict(x, verbose=0)[0][0])
            if prob >= CNN_THRESHOLD:
                cnn_streak += 1
            else:
                cnn_streak = 0

        # ---- Eye fallback ----
        if eyes_closed(rgb):
            eye_streak += 1
        else:
            eye_streak = 0

        # ---- Status ----
        trigger = None
        if cnn_streak >= CNN_STREAK_N:
            trigger = "CNN"
        if eye_streak >= EYE_STREAK_N:
            trigger = "EYES"

        if trigger:
            status = f"DROWSY ({trigger})"
        else:
            status = "AWAKE"

        ptxt = f"{prob:.3f}" if prob is not None else "warming"
        print(f"prob={ptxt}  cnn={cnn_streak:02d}  eyes={eye_streak:02d}  {status}")

        # ---- Alarm ----
        if trigger and (time.time() - last_alarm) > ALARM_GAP:
            print("🚨 ALARM!")
            alarm()
            last_alarm = time.time()
            cnn_streak = 0
            eye_streak = 0

        time.sleep(0.05)

except KeyboardInterrupt:
    print("Stopped")

Model expects: SEQ_LEN=16, H=64, W=64, C=3



LIVE DETECTION RUNNING

prob=warming  cnn=00  eyes=00  AWAKE
prob=warming  cnn=00  eyes=00  AWAKE
prob=warming  cnn=00  eyes=00  AWAKE
prob=warming  cnn=00  eyes=00  AWAKE
prob=warming  cnn=00  eyes=00  AWAKE
prob=warming  cnn=00  eyes=00  AWAKE
prob=warming  cnn=00  eyes=00  AWAKE
prob=warming  cnn=00  eyes=00  AWAKE
prob=warming  cnn=00  eyes=00  AWAKE
prob=warming  cnn=00  eyes=00  AWAKE
prob=warming  cnn=00  eyes=00  AWAKE
prob=warming  cnn=00  eyes=00  AWAKE
prob=warming  cnn=00  eyes=00  AWAKE
prob=warming  cnn=00  eyes=00  AWAKE
prob=warming  cnn=00  eyes=00  AWAKE
prob=0.392  cnn=00  eyes=00  AWAKE
prob=0.400  cnn=00  eyes=00  AWAKE
prob=0.403  cnn=00  eyes=00  AWAKE
prob=0.403  cnn=00  eyes=00  AWAKE
prob=0.404  cnn=00  eyes=00  AWAKE
prob=0.401  cnn=00  eyes=00  AWAKE
prob=0.399  cnn=00  eyes=00  AWAKE
prob=0.401  cnn=00  eyes=00  AWAKE
prob=0.403  cnn=00  eyes=00  AWAKE
prob=0.403  cnn=00  eyes=00  AWAKE
prob=0.402  cnn=00  eyes=00  AWAKE
prob=0.404  cnn=00  eyes=00  AWAKE


AttributeError: 'NoneType' object has no attribute 'split'